# 64 - Multi-Position Strategy: Aggressive Entry, Gradual Exit

**Concept:**
- 🟢 **Aggressive BUY:** Open multiple positions when signal is bullish
- 🔴 **Gradual SELL:** Scale out slowly as signal deteriorates

This mimics how professional traders manage positions:
1. Build a position in tranches during accumulation
2. Take profits in tranches during distribution
3. Never fully exit - always keep a "moon bag"

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Optional
import warnings
warnings.filterwarnings('ignore')

plt.style.use('dark_background')
plt.rcParams['figure.figsize'] = (14, 6)

DATA_DIR = Path.home() / "Documents" / "bitcoin-lab-btc-data-pipeline" / "data" / "daily"

def load_metric(name):
    path = DATA_DIR / f"{name}.parquet"
    if not path.exists():
        return pd.DataFrame(columns=['time', 'value'])
    df = pd.read_parquet(path)
    if 'time' in df.columns:
        df['time'] = pd.to_datetime(df['time'])
        if df['time'].dt.tz is not None:
            df['time'] = df['time'].dt.tz_localize(None)
        df = df.set_index('time').sort_index()
    return df

data = {m: load_metric(m) for m in ['price', 'mvrv', 'mvrv_sth', 'mvrv_lth', 'nupl', 'sopr', 'aviv']}
print("Data loaded")

In [ ]:
# Build composite signal
CONFIG = {
    'mvrv': {'weight': 0.30, 'bullish': 1.0, 'bearish': 2.4},
    'mvrv_sth': {'weight': 0.15, 'bullish': 1.0, 'bearish': 1.4},
    'mvrv_lth': {'weight': 0.15, 'bullish': 1.5, 'bearish': 3.5},
    'nupl': {'weight': 0.20, 'bullish': 0.25, 'bearish': 0.6},
    'sopr': {'weight': 0.10, 'bullish': 1.0, 'bearish': 1.05},
    'aviv': {'weight': 0.10, 'bullish': 1.0, 'bearish': 1.5},
}

def score_metric(value, bullish, bearish):
    if pd.isna(value): return 0
    midpoint = (bullish + bearish) / 2
    if value <= bullish:
        return -1 - (bullish - value) / bullish
    elif value <= midpoint:
        return -1 + (value - bullish) / (midpoint - bullish)
    elif value <= bearish:
        return (value - midpoint) / (bearish - midpoint)
    else:
        return 1 + min((value - bearish) / bearish, 1)

df = data['price'][['value']].rename(columns={'value': 'price'}).copy()
df['returns'] = df['price'].pct_change()

for metric in CONFIG.keys():
    if metric in data and not data[metric].empty:
        df = df.join(data[metric][['value']].rename(columns={'value': metric}), how='left')
        df[metric] = df[metric].ffill()

def calc_composite(row):
    total_score, total_weight = 0, 0
    for metric, cfg in CONFIG.items():
        if metric in row and pd.notna(row[metric]):
            total_score += score_metric(row[metric], cfg['bullish'], cfg['bearish']) * cfg['weight']
            total_weight += cfg['weight']
    return total_score / total_weight if total_weight > 0 else 0

df['signal'] = df.apply(calc_composite, axis=1)
print(f"Data: {df.index[0].date()} to {df.index[-1].date()}")

---
## Position/Tranche Tracking System

In [ ]:
@dataclass
class Position:
    """Represents a single position/tranche."""
    entry_date: pd.Timestamp
    entry_price: float
    size: float
    position_id: int
    
    def current_value(self, current_price: float) -> float:
        return self.size * (current_price / self.entry_price)
    
    def pnl_pct(self, current_price: float) -> float:
        return (current_price / self.entry_price - 1) * 100

@dataclass 
class PortfolioManager:
    """Manages multiple positions with entry/exit logic."""
    initial_capital: float = 100000
    max_positions: int = 10
    position_size: float = 0.10
    
    positions: List[Position] = field(default_factory=list)
    cash: float = field(init=False)
    position_counter: int = 0
    history: List[dict] = field(default_factory=list)
    trades: List[dict] = field(default_factory=list)
    
    def __post_init__(self):
        self.cash = self.initial_capital
    
    def total_value(self, current_price):
        return self.cash + sum(p.current_value(current_price) * self.initial_capital for p in self.positions)
    
    def invested_pct(self, current_price):
        total = self.total_value(current_price)
        return 1 - (self.cash / total) if total > 0 else 0
    
    def can_open_position(self):
        return len(self.positions) < self.max_positions and self.cash >= self.initial_capital * self.position_size
    
    def open_position(self, date, price, size=None):
        size = size or self.position_size
        cost = self.initial_capital * size
        if self.cash < cost: return None
        self.position_counter += 1
        position = Position(entry_date=date, entry_price=price, size=size, position_id=self.position_counter)
        self.positions.append(position)
        self.cash -= cost
        self.trades.append({'date': date, 'action': 'BUY', 'price': price, 'size': size, 'position_id': position.position_id})
        return position
    
    def close_position(self, position, date, price):
        if position not in self.positions: return
        value = position.current_value(price) * self.initial_capital
        pnl = position.pnl_pct(price)
        self.cash += value
        self.positions.remove(position)
        self.trades.append({'date': date, 'action': 'SELL', 'price': price, 'size': position.size,
                           'position_id': position.position_id, 'pnl_pct': pnl,
                           'entry_price': position.entry_price, 'hold_days': (date - position.entry_date).days})
    
    def close_most_profitable(self, date, price):
        if self.positions:
            best = max(self.positions, key=lambda p: p.pnl_pct(price))
            self.close_position(best, date, price)
    
    def close_by_profit_target(self, date, price, target_pct):
        to_close = [p for p in self.positions if p.pnl_pct(price) >= target_pct]
        for p in to_close: self.close_position(p, date, price)
        return len(to_close)
    
    def record_state(self, date, price, signal):
        self.history.append({'date': date, 'price': price, 'signal': signal,
                            'total_value': self.total_value(price), 'cash': self.cash,
                            'invested_pct': self.invested_pct(price), 'num_positions': len(self.positions)})

print("Portfolio manager defined")

---
## Strategy 1: Signal Entry + Profit Target Exit

In [ ]:
def run_profit_target_strategy(df, buy_signal=-0.5, aggressive_buy=-1.0, profit_targets=[50, 100, 200]):
    pm = PortfolioManager(initial_capital=100000, max_positions=10, position_size=0.10)
    for date, row in df.iterrows():
        price, signal = row['price'], row['signal']
        if pd.isna(price): continue
        for target in profit_targets: pm.close_by_profit_target(date, price, target)
        if signal <= aggressive_buy:
            for _ in range(2):
                if pm.can_open_position(): pm.open_position(date, price)
        elif signal <= buy_signal:
            if pm.can_open_position(): pm.open_position(date, price)
        pm.record_state(date, price, signal)
    return pm

pm_profit = run_profit_target_strategy(df)
print(f"Profit Target: {len(pm_profit.trades)} trades, Final: ${pm_profit.history[-1]['total_value']:,.0f}")

---
## Strategy 2: Signal Entry + Signal Exit (with Moon Bag)

In [ ]:
def run_signal_exit_strategy(df, buy_signal=-0.5, aggressive_buy=-1.0, sell_signal=0.75, aggressive_sell=1.5, moon_bag=2):
    pm = PortfolioManager(initial_capital=100000, max_positions=10, position_size=0.10)
    for date, row in df.iterrows():
        price, signal = row['price'], row['signal']
        if pd.isna(price): continue
        if signal >= aggressive_sell:
            while len(pm.positions) > moon_bag: pm.close_most_profitable(date, price)
        elif signal >= sell_signal:
            if len(pm.positions) > moon_bag: pm.close_most_profitable(date, price)
        if signal <= aggressive_buy:
            for _ in range(2):
                if pm.can_open_position(): pm.open_position(date, price)
        elif signal <= buy_signal:
            if pm.can_open_position(): pm.open_position(date, price)
        pm.record_state(date, price, signal)
    return pm

pm_signal = run_signal_exit_strategy(df)
print(f"Signal Exit: {len(pm_signal.trades)} trades, Final: ${pm_signal.history[-1]['total_value']:,.0f}")

---
## Strategy 3: DCA + Signal Boost

In [ ]:
def run_dca_boost_strategy(df, dca_interval=14, base_size=0.03, boost_signal=-0.5, super_boost=-1.0, sell_signal=1.5):
    pm = PortfolioManager(initial_capital=100000, max_positions=100, position_size=base_size)
    last_dca = None
    for date, row in df.iterrows():
        price, signal = row['price'], row['signal']
        if pd.isna(price): continue
        if signal >= sell_signal: pm.close_by_profit_target(date, price, 100)
        is_dca_day = last_dca is None or (date - last_dca).days >= dca_interval
        if is_dca_day and pm.invested_pct(price) < 0.90:
            if signal <= super_boost: size = base_size * 3
            elif signal <= boost_signal: size = base_size * 2
            elif signal >= sell_signal: size = 0
            else: size = base_size
            if size > 0 and pm.cash >= pm.initial_capital * size:
                pm.open_position(date, price, size)
                last_dca = date
        pm.record_state(date, price, signal)
    return pm

pm_dca = run_dca_boost_strategy(df)
print(f"DCA Boost: {len(pm_dca.trades)} trades, Final: ${pm_dca.history[-1]['total_value']:,.0f}")

---
## Strategy 4: Layered Entry + Trailing Exit

In [ ]:
def run_layered_trailing_strategy(df, layers=[(-0.5, 0.15), (-1.0, 0.20), (-1.5, 0.25)],
                                   trail_pct=0.30, min_profit=0.75, moon_bag_pct=0.15):
    pm = PortfolioManager(initial_capital=100000, max_positions=20, position_size=0.10)
    peak_prices, layers_used, last_bullish = {}, set(), False
    for date, row in df.iterrows():
        price, signal = row['price'], row['signal']
        if pd.isna(price): continue
        bullish = signal < 0
        if bullish and not last_bullish: layers_used = set()
        last_bullish = bullish
        for p in pm.positions: peak_prices[p.position_id] = max(peak_prices.get(p.position_id, price), price)
        moon_bag_value = pm.initial_capital * moon_bag_pct
        to_close = []
        for p in pm.positions:
            profit = p.pnl_pct(price) / 100
            peak = peak_prices.get(p.position_id, price)
            dd = (peak - price) / peak if peak > 0 else 0
            if profit > min_profit and dd > trail_pct:
                invested = sum(pos.current_value(price) * pm.initial_capital for pos in pm.positions)
                if invested > moon_bag_value: to_close.append(p)
        for p in to_close: pm.close_position(p, date, price)
        for i, (thresh, size) in enumerate(layers):
            if i not in layers_used and signal <= thresh:
                if pm.cash >= pm.initial_capital * size:
                    pm.open_position(date, price, size)
                    layers_used.add(i)
        pm.record_state(date, price, signal)
    return pm

pm_layered = run_layered_trailing_strategy(df)
print(f"Layered Trailing: {len(pm_layered.trades)} trades, Final: ${pm_layered.history[-1]['total_value']:,.0f}")

---
## Compare All Strategies

In [ ]:
def analyze_pm(pm, name):
    hist = pd.DataFrame(pm.history).set_index('date')
    hist['returns'] = hist['total_value'].pct_change()
    hist['dd'] = hist['total_value'] / hist['total_value'].cummax() - 1
    years = (hist.index[-1] - hist.index[0]).days / 365
    sells = [t for t in pm.trades if t['action'] == 'SELL']
    avg_profit = np.mean([t.get('pnl_pct', 0) for t in sells]) if sells else 0
    win_rate = len([t for t in sells if t.get('pnl_pct', 0) > 0]) / len(sells) * 100 if sells else 0
    return {'name': name, 'cagr': ((hist['total_value'].iloc[-1] / 100000) ** (1/years) - 1) * 100,
            'max_dd': hist['dd'].min() * 100, 'sharpe': (hist['returns'].mean() / hist['returns'].std()) * np.sqrt(365) if hist['returns'].std() > 0 else 0,
            'trades': len(pm.trades), 'avg_profit': avg_profit, 'win_rate': win_rate,
            'equity': hist['total_value'], 'dd': hist['dd'], 'invested': hist['invested_pct']}

hodl = 100000 * (1 + df['returns']).cumprod()
hodl_dd = hodl / hodl.cummax() - 1
years = (df.index[-1] - df.index[0]).days / 365

results = [
    {'name': 'HODL', 'cagr': ((hodl.iloc[-1]/100000)**(1/years)-1)*100, 'max_dd': hodl_dd.min()*100,
     'sharpe': (df['returns'].mean()/df['returns'].std())*np.sqrt(365), 'trades': 1,
     'avg_profit': 0, 'win_rate': 0, 'equity': hodl, 'dd': hodl_dd, 'invested': pd.Series(1.0, index=df.index)},
    analyze_pm(pm_profit, 'Profit Target'),
    analyze_pm(pm_signal, 'Signal Exit'),
    analyze_pm(pm_dca, 'DCA Boost'),
    analyze_pm(pm_layered, 'Layered Trail'),
]

print(f"{'Strategy':<18} {'CAGR':>8} {'MaxDD':>8} {'Sharpe':>8} {'Trades':>8}")
print("-"*60)
for r in results:
    print(f"{r['name']:<18} {r['cagr']:>7.1f}% {r['max_dd']:>7.1f}% {r['sharpe']:>8.2f} {r['trades']:>8}")

## Summary

**Multi-Position Approach:**
1. **Aggressive Entry** - Open multiple positions when signal is bullish
2. **Gradual Exit** - Scale out as signal becomes bearish  
3. **Moon Bag** - Always keep 1-2 positions for asymmetric upside